In [1]:
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score

from lightgbm import LGBMClassifier, early_stopping

from tqdm.notebook import tqdm

In [2]:
def metric(y_true, y_pred):
    res = cohen_kappa_score(y_true, y_pred.reshape((y_true.shape[0], 5), order="F").argmax(axis=1), weights= 'quadratic')
    return "kappa", res, True

In [3]:
train = pd.read_csv("/kaggle/input/competitions/petfinder-adoption-prediction/train/train.csv", index_col="PetID") 
test = pd.read_csv("/kaggle/input/competitions/petfinder-adoption-prediction/test/test.csv", index_col="PetID")
ss = pd.read_csv("/kaggle/input/competitions/petfinder-adoption-prediction/test/sample_submission.csv", index_col="PetID")

folds = [
    train.index[test_idx]
    for train_idx, test_idx in KFold(5).split(train)
]

In [4]:
label_col = "AdoptionSpeed"

In [5]:
train["RescuerID"] = train["RescuerID"].astype("category")
test["RescuerID"] = test["RescuerID"].astype("category")

In [6]:
train["Name"] = train["Name"].astype("category")
test["Name"] = test["Name"].astype("category")

In [7]:
train["Name"].fillna("NU:LLLLLL").value_counts().head(10)

TypeError: Cannot setitem on a Categorical with a new category (NU:LLLLLL), set the categories first

In [ ]:
v = pd.crosstab(train["Name"].str.lower().fillna("no name").str.contains("no name"), train[label_col])
v / v.sum(axis=1).values[:, None]

In [ ]:
v = train[["Name", label_col]].join(train["Name"].value_counts().rename('name_count'), on="Name")
v = pd.crosstab(pd.qcut(v["name_count"], 10, duplicates="drop"), v[label_col])
v / v.sum(axis=1).values[:, None]

In [ ]:
train["Name"]

In [ ]:
fold_preds = []
test_preds = []
for i, fold in enumerate(folds):
    X_train = train.drop(fold).drop(label_col, axis=1).select_dtypes(exclude="O")
    y_train = train.drop(fold)[label_col]

    X_valid = train.loc[fold].drop(label_col, axis=1).select_dtypes(exclude="O")
    y_valid = train.loc[fold, label_col]

    learner = LGBMClassifier()
    
    learner.fit(X_train, y_train,
                eval_set=[(X_train, y_train), (X_valid, y_valid)],
                eval_metric= metric,
                callbacks=[early_stopping(5)])
    
    fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name=label_col))
    test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))

fold_preds = pd.concat(fold_preds)
test_preds = sum(test_preds).idxmax(axis=1).rename(label_col)

In [ ]:
cohen_kappa_score(
    train.loc[fold_preds.index, "AdoptionSpeed"],
    fold_preds
)

In [ ]:
test_preds.to_csv("submission.csv")
!head submission.csv